# Quiz 1 Prep: Python + Stats + Plots + PCA

This notebook gives you **practice code** for everything listed in the study guide:
- Load the dataset with pandas
- Quick data checks (shape, missing values)
- Grouping and summaries (mean, std, `groupby().agg()`)
- Plots:
  1. Scatter: **payroll vs. winning percentage** (overall + by time period)
  2. Line: **payroll trends over time** for selected teams
  3. Boxplot: **payroll distributions by year** (interpret the median & spread)
- A **short PCA explanation** + basic PCA plots

> Expected CSV columns: `team, year, payroll, win_num, win_pct`  
> Rows: one per **team-season** (1998–2014).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

# For PCA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Display options
pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", None)


In [ ]:
# ---- Load the dataset ----
# Place your real file as "baseball.csv" in the same folder as this notebook (or in /mnt/data if you're running here).
# If not found, we'll create a small demo dataset so that all code runs end-to-end.

DATA_PATHS = [Path("baseball.csv"), Path("/mnt/data/baseball.csv")]
df = None
for p in DATA_PATHS:
    if p.exists():
        df = pd.read_csv(p)
        print(f"Loaded data from: {p.resolve()}")
        break

if df is None:
    # Create a simple demo dataset (1998–2014, 6 teams) so plots & PCA still work.
    rng = np.random.default_rng(42)
    teams = ["Yankees", "Red Sox", "Dodgers", "Athletics", "Rays", "Marlins"]
    years = list(range(1998, 2015))
    rows = []
    for y in years:
        for t in teams:
            # make payroll vary by team with noise
            base = {
                "Yankees": 180, "Dodgers": 150, "Red Sox": 140,
                "Athletics": 60, "Rays": 55, "Marlins": 50
            }[t]
            payroll = base + rng.normal(0, 15)
            # wins around 81 with slight boost for high payroll teams
            win_num = int(np.clip(70 + (payroll - 90)/6 + rng.normal(0, 6), 50, 110))
            win_pct = win_num / 162
            rows.append([t, y, round(float(payroll), 2), win_num, round(float(win_pct), 3)])
    df = pd.DataFrame(rows, columns=["team", "year", "payroll", "win_num", "win_pct"])
    demo_path = Path("baseball_demo.csv")
    df.to_csv(demo_path, index=False)
    print("⚠️ 'baseball.csv' not found. Generated a demo dataset and saved as:", demo_path.resolve())

# Ensure correct dtypes
df["year"] = df["year"].astype(int)
df.head()


In [ ]:
# ---- Data checks ----
print("Shape:", df.shape)          # (rows, columns)
print("\nMissing values by column:\n", df.isnull().sum())

# Basic describe for numeric columns
df.describe(numeric_only=True)


In [ ]:
# ---- Grouping and aggregation ----
# Example 1: league-wide means & std
league_means = df[["payroll", "win_num", "win_pct"]].mean()
league_stds  = df[["payroll", "win_num", "win_pct"]].std()
print("League-wide means:\n", league_means, "\n")
print("League-wide stds:\n", league_stds, "\n")

# Example 2: per-team summaries across all years
team_summary = (
    df.groupby("team")
      .agg(
          seasons=("year", "nunique"),
          avg_payroll=("payroll", "mean"),
          avg_win_pct=("win_pct", "mean"),
          total_wins=("win_num", "sum")
      )
      .sort_values("avg_payroll", ascending=False)
      .round(3)
)
team_summary.head(10)


In [ ]:
# ---- Scatterplot: payroll vs win_pct (overall) ----
plt.figure(figsize=(6, 4))
plt.scatter(df["payroll"], df["win_pct"], alpha=0.6)
plt.xlabel("Payroll (Millions)")
plt.ylabel("Winning Percentage")
plt.title("Payroll vs. Winning Percentage (All Years)")
plt.grid(True, linestyle="--", linewidth=0.5)
plt.show()


In [ ]:
# ---- Scatterplot: payroll vs win_pct by time period ----
# You can adjust the time splits if you want (e.g., early vs late period)
period_1 = df[df["year"] <= 2005]
period_2 = df[df["year"] >= 2006]

plt.figure(figsize=(6, 4))
plt.scatter(period_1["payroll"], period_1["win_pct"], alpha=0.6, label="≤ 2005")
plt.scatter(period_2["payroll"], period_2["win_pct"], alpha=0.6, label="≥ 2006")
plt.xlabel("Payroll (Millions)")
plt.ylabel("Winning Percentage")
plt.title("Payroll vs. Winning Percentage by Period")
plt.legend()
plt.grid(True, linestyle="--", linewidth=0.5)
plt.show()


In [ ]:
# ---- Line plot: payroll trends for selected teams ----
# We'll pick 2 teams: one high-spender and one low-spender based on avg payroll
top_team = team_summary.index[0]
bottom_team = team_summary.index[-1]

print("Top-spend example team:", top_team)
print("Low-spend example team:", bottom_team)

def plot_team_payroll(team_name: str):
    sub = df[df["team"] == team_name].sort_values("year")
    plt.figure(figsize=(6, 4))
    plt.plot(sub["year"], sub["payroll"], marker="o")
    plt.xlabel("Year")
    plt.ylabel("Payroll (Millions)")
    plt.title(f"Payroll Trend: {team_name}")
    plt.grid(True, linestyle="--", linewidth=0.5)
    plt.show()

plot_team_payroll(top_team)
plot_team_payroll(bottom_team)


In [ ]:
# ---- Boxplot: payroll distribution by year ----
# Matplotlib boxplot expects a list of arrays for each year
years_sorted = sorted(df["year"].unique())
data_by_year = [df.loc[df["year"] == y, "payroll"].values for y in years_sorted]

plt.figure(figsize=(10, 4))
plt.boxplot(data_by_year, labels=years_sorted, showfliers=False)
plt.xlabel("Year")
plt.ylabel("Payroll (Millions)")
plt.title("Payroll Distribution by Year (Boxplot)")
plt.xticks(rotation=45)
plt.grid(True, axis="y", linestyle="--", linewidth=0.5)
plt.tight_layout()
plt.show()


## PCA: Short Explanation (Read This)

**Principal Component Analysis (PCA)** creates new variables (principal components) as **linear combinations** of your original features.  
Each component’s **loadings** (one weight per original feature) define its direction in feature space, and these directions are **unique up to a sign flip** (multiplying a component and all its loadings by −1 leaves it equivalent).  
In a standard **PC1 vs. PC2 plot**, each point’s **scores** are its coordinates on those components—so points near each other have similar combinations of the original variables. **Loadings** tell you which original features contribute most to each component: large-magnitude loadings mean that feature is important in that component. Interpreting PCA means reading **variance explained** by each PC, the **scores** pattern (clusters, gradients), and the **loadings** to connect components back to meaningful directions in the original variables.


In [ ]:
# ---- PCA on numeric features ----
numeric = df[["payroll", "win_num", "win_pct"]].copy()

# Standardize features (important for PCA)
scaler = StandardScaler()
Z = scaler.fit_transform(numeric)

pca = PCA(n_components=3, random_state=0)
Z_pca = pca.fit_transform(Z)

explained = pca.explained_variance_ratio_
loadings = pca.components_.T  # shape: (n_features, n_components)
features = numeric.columns.tolist()

print("Explained variance ratio per PC:", np.round(explained, 4))
print("\nLoadings (rows: features, cols: PC):\n")
load_df = pd.DataFrame(loadings, index=features, columns=[f"PC{i+1}" for i in range(loadings.shape[1]]])
load_df.round(3)


In [ ]:
# ---- Plot: Explained variance ratio (bar) ----
plt.figure(figsize=(6, 4))
x = np.arange(1, len(explained) + 1)
plt.bar(x, explained)
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("PCA: Explained Variance")
plt.xticks(x, [f"PC{i}" for i in x])
plt.grid(True, axis="y", linestyle="--", linewidth=0.5)
plt.show()


In [ ]:
# ---- Plot: PC1 vs PC2 scores ----
plt.figure(figsize=(6, 4))
plt.scatter(Z_pca[:, 0], Z_pca[:, 1], alpha=0.6)
plt.xlabel("PC1 (scores)")
plt.ylabel("PC2 (scores)")
plt.title("PCA Scores: PC1 vs PC2")
plt.grid(True, linestyle="--", linewidth=0.5)
plt.show()


In [ ]:
# ---- Optional: simple "biplot" style overlay of loadings (scaled for visibility) ----
# (Pure matplotlib, minimal; arrows show feature directions)
plt.figure(figsize=(6, 4))
plt.scatter(Z_pca[:, 0], Z_pca[:, 1], alpha=0.5)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA Scores with Loadings Overlay")

# scale arrows to look reasonable
scale = 2.5
for i, feat in enumerate(features):
    plt.arrow(0, 0, loadings[i, 0]*scale, loadings[i, 1]*scale, head_width=0.05, length_includes_head=True)
    plt.text(loadings[i, 0]*scale*1.1, loadings[i, 1]*scale*1.1, feat)

plt.grid(True, linestyle="--", linewidth=0.5)
plt.show()
